# Module 03 lab: Leduc Hold'em

Work through the chance nodes, betting rules, information sets and solver variants. The state interface is intentionally small; exact full-tree solving can be slow, so start with low iteration counts.

In [ ]:
from pokerbot.games.leduc import LeducPoker, card_name

game = LeducPoker()
root = game.root()
deals = root.chance_outcomes()
len(deals), sum(p for _, p in deals), [(tuple(card_name(c) for c in d), p) for d, p in deals[:3]]

## Inspect the public card chance node

After two checks, betting round one ends. The public card is drawn from the four cards not in the private deal.

In [ ]:
private, probability = deals[0]
after_checks = root.child(private).child('c').child('c')
assert after_checks.is_chance()
[(card_name(card), p) for card, p in after_checks.chance_outcomes()]

## Look at information sets

Change only the hidden opponent card. If the acting player's rank, board and public history are unchanged, their information-set key should remain unchanged.

In [ ]:
state_a = game.root().child((0, 2))
state_b = game.root().child((0, 4))
state_a.info_set_key(0), state_b.info_set_key(0)

## Compare solver variants

Run the same small iteration budget for each configuration. Then increase the budget carefully and record both wall time and exact exploitability. An iteration count alone does not measure speed.

In [ ]:
from time import perf_counter
from pokerbot.solvers.cfr import CFRTrainer
from pokerbot.eval.exploitability import exploitability

variants = {
    'vanilla': {},
    'CFR+': {'plus': True},
    'alternating CFR+ with linear average': {
        'plus': True, 'alternating': True, 'linear_average': True
    },
}
results = {}
for name, options in variants.items():
    trainer = CFRTrainer(game, **options)
    start = perf_counter()
    trainer.train(2)  # increase after measuring runtime
    elapsed = perf_counter() - start
    average = trainer.average_strategy()
    results[name] = {'seconds': elapsed, 'infosets': len(average),
                     'exploitability': exploitability(game, average)}
results